# 04 Closed classes

Notebook 03 found that the α^k bound is nearly attained when the chain starts
from a single page, and loose when it starts from the uniform distribution. This
notebook looks for the reason in the link structure.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.sparse.csgraph import connected_components
from src.load import load_graph
from src.chain import build_chain

A, node_ids, info = load_graph()
PT, dangling = build_chain(A)
n = A.shape[0]
pi = np.load("../data/pi_alpha_0.85.npy")

rng = np.random.default_rng(0)
top_page = int(np.argmax(pi))
random_page = int(rng.integers(n))

## Strongly connected components

Two pages are in the same strongly connected component when each can be reached
from the other by following links. Every closed class is one of these components.

In [2]:
n_components, labels = connected_components(A, directed=True, connection="strong")
sizes = np.bincount(labels)
print("strongly connected components:", n_components)
print("largest component:", sizes.max())

strongly connected components: 29914
largest component: 150532


## Closed classes

A component is closed when no link leaves it. Dangling pages are excluded.

In [3]:
edges = A.tocoo()
crossing = labels[edges.row] != labels[edges.col]

has_exit = np.zeros(n_components, dtype=bool)
has_exit[labels[edges.row[crossing]]] = True

has_dangling = np.zeros(n_components, dtype=bool)
has_dangling[labels[dangling]] = True

closed = ~has_exit & ~has_dangling
in_closed = closed[labels]

print("closed classes:", closed.sum())
print("pages in closed classes:", in_closed.sum())
print("largest closed class:", sizes[closed].max())

closed classes: 2231
pages in closed classes: 85261
largest closed class: 4888


In [4]:
(pd.Series(sizes[closed]).value_counts().sort_index()
   .rename_axis("class size").rename("number of classes").to_frame().head(12))

,number of classes
class size,
2,370
3,159
4,124
5,93
6,52
7,69
8,94
9,82
10,34


The graph has 2,231 closed classes with 85,261 pages (30% of the total). 

Haveliwala and Kamvar (2003) show that when the link structure has two or more
closed classes, the second eigenvalue of the chain has modulus exactly α. Since that is true here, the long-run rate of convergence is exactly α.

In [5]:
print("pi mass inside closed classes (alpha = 0.85):", pi[in_closed].sum())
print("top page is in a closed class:", bool(in_closed[top_page]),
      " class size:", sizes[labels[top_page]])
print("random page is in a closed class:", bool(in_closed[random_page]),
      " class size:", sizes[labels[random_page]])

pi mass inside closed classes (alpha = 0.85): 0.33918514177969744
top page is in a closed class: False  class size: 150532
random page is in a closed class: False  class size: 150532


At α = 0.85, 33.9% of the stationary mass lies inside closed classes. 

## Surfer without teleporting

By the identity behind the bound, the ratio r_k equals the TV distance between
two link-only surfers after k steps divided by their starting distance. We run the chain with α = 1 to compute it.

In [6]:
ALPHAS = [0.50, 0.70, 0.85, 0.95, 0.99]
rows = []
for alpha in ALPHAS:
    p_alpha = np.load(f"../data/pi_alpha_{alpha:.2f}.npy")
    rows.append({"alpha": alpha, "pi mass in closed classes": p_alpha[in_closed].sum()})
pd.DataFrame(rows)

,alpha,pi mass in closed classes
0,0.50,0.312863
1,0.70,0.321946
2,0.85,0.339185
3,0.95,0.386639
4,0.99,0.520065


Closed classes hold 30% of the pages. They hold 31% of π at α = 0.50 and 52% at α = 0.99, which makes sense since the less often the surfer teleports, the more rank collects in sets it cannot leave.

In [7]:
from src.chain import step
from src.metrics import tv


def point_mass(i):
    x = np.zeros(n)
    x[i] = 1.0
    return x


starts = {
    "uniform": np.full(n, 1.0 / n),
    "top page": point_mass(top_page),
    "random page": point_mass(random_page),
}

CHECKPOINTS = [10, 100, 1000, 2000, 5000, 10000]

x = {name: x0.copy() for name, x0 in starts.items()}
p = pi.copy()
tv0 = {name: tv(x0, pi) for name, x0 in starts.items()}

rows = []
for k in range(1, max(CHECKPOINTS) + 1):
    p = step(p, PT, dangling, 1.0)
    for name in x:
        x[name] = step(x[name], PT, dangling, 1.0)
    if k in CHECKPOINTS:
        row = {"steps": k, "pi mass trapped": p[in_closed].sum()}
        for name in x:
            row[name] = tv(x[name], p) / tv0[name]
        rows.append(row)

link_only = pd.DataFrame(rows)
link_only

,steps,pi mass trapped,uniform,top page,random page
0,10,0.385064,0.116985,0.906468,0.848612
1,100,0.570188,0.023994,0.858724,0.585102
2,1000,0.860991,0.005973,0.831930,0.479111
3,2000,0.909694,0.005085,0.829635,0.464711
4,5000,0.943144,0.004800,0.827716,0.454790
5,10000,0.950963,0.004761,0.827258,0.451947


The ratio levels off for every start, which means that in the long run the distance behaves like c × α^k times the starting distance. The rate α is the same for all three starts. The constant c is 0.83 from the top page, 0.45 from the random page, and 0.0048 from the uniform distribution. At α = 0.85 that puts the uniform start about 33 steps ahead of the bound, and the top page about one step ahead.

In [13]:
conv = pd.read_csv("../data/convergence.csv")
conv[(conv["alpha"] == 0.85) & (conv["k"] == 100)][["start", "ratio"]]

,start,ratio
375,uniform,0.023994
490,top page,0.858724
631,random page,0.585102


At 100 steps these ratios agree with the ones measured in notebook 03 to six digits, so our identity seems to hold.

In [10]:
def class_mass(v):
    return np.bincount(labels, weights=v, minlength=n_components)[closed]


def summary(name, v):
    ranked = np.sort(class_mass(v))[::-1]
    return {
        "start": name,
        "mass trapped": ranked.sum(),
        "largest class share": ranked[0],
        "classes holding 90% of trapped mass": int(np.searchsorted(np.cumsum(ranked), 0.9 * ranked.sum()) + 1),
    }


pd.DataFrame([summary(name, v) for name, v in x.items()] + [summary("pi", p)])

,start,mass trapped,largest class share,classes holding 90% of trapped mass
0,uniform,0.950960,0.052913,754
1,top page,0.999211,0.359118,4
2,random page,0.980593,0.063886,162
3,pi,0.950963,0.052913,754


After 10,000 steps without teleporting, at least 95% of every surfer's mass is inside closed classes. The top page is not in a closed class, but its links lead into very few. The mass of π is spread over 754.

The uniform start ends with the same distribution over classes as π.

## Why the uniform start does not reach zero

Rearranging π = απS + (1 − α)u gives

$$\pi = (1-\alpha)\sum_{j \ge 0} \alpha^{j}\, u S^{j}$$

so π is a weighted average of the uniform start's link-only path. Both surfers therefore carry the same mass into every closed class, and any distance that remains must lie inside the classes.

In [11]:
gap = 0.5 * np.abs(x["uniform"] - p) / tv0["uniform"]
is_pair = in_closed & (sizes[labels] == 2)

pd.Series({
    "total ratio": gap.sum(),
    "between classes": 0.5 * np.abs(class_mass(x["uniform"]) - class_mass(p)).sum() / tv0["uniform"],
    "inside closed classes": gap[in_closed].sum(),
    "  of which inside two-page classes": gap[is_pair].sum(),
    "outside closed classes": gap[~in_closed].sum(),
})

total ratio                           0.004761
between classes                       0.000003
inside closed classes                 0.004758
  of which inside two-page classes    0.000608
outside closed classes                0.000003
dtype: float64

99.9% of the uniform start's remaining distance is inside closed classes, and almost none is between them.

In [14]:
WINDOW = 2520

u_k, p_k = x["uniform"].copy(), p.copy()
u_avg, p_avg = np.zeros(n), np.zeros(n)
for _ in range(WINDOW):
    u_k = step(u_k, PT, dangling, 1.0)
    p_k = step(p_k, PT, dangling, 1.0)
    u_avg += u_k / WINDOW
    p_avg += p_k / WINDOW

print("ratio at one step:     ", tv(u_k, p_k) / tv0["uniform"])
print("ratio of the averages: ", tv(u_avg, p_avg) / tv0["uniform"])

ratio at one step:      0.004755587930841165
ratio of the averages:  4.9930888637747095e-06


Notice that veraging both surfers over 2,520 consecutive steps reduces the ratio from 0.0048 to 0.000005. The remaining distance is a disjoint in phase inside periodic classes, where the surfers agree on how much mass each class holds and disagree on where in its cycle it is. If every closed class were aperiodic, the uniform start would converge faster than rate α.